In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read bronze table
bronze_df = spark.table("first_data_engineering_project.bronze.bronze_orders")

print(f"Bronze table row count: {bronze_df.count()}")

# Apply silver transformations
silver_df = (
    bronze_df
    
    # Remove records with NULL values in critical columns
    .filter(
        F.col("order_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        F.col("store_id").isNotNull() &
        F.col("order_date").isNotNull()
    )
    
    # Remove duplicate orders (keep the most recent by ingestion_timestamp)
    .withColumn(
        "row_num",
        F.row_number().over(
            Window.partitionBy("order_id")
            .orderBy(F.col("ingestion_timestamp").desc())
        )
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    
    # Validate data quality - ensure positive IDs
    .filter(
        (F.col("order_id") > 0) &
        (F.col("customer_id") > 0) &
        (F.col("store_id") > 0)
    )
    
    # Add silver layer metadata
    .withColumn("silver_processed_timestamp", F.current_timestamp())
    .withColumn("data_quality_flag", F.lit("VALID"))
)

print(f"Silver table row count after transformations: {silver_df.count()}")

# Write to silver table
(
    silver_df
    .write
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("first_data_engineering_project.silver.silver_orders")
)

print("✓ Silver table created successfully: first_data_engineering_project.silver.silver_orders")

# Display sample of silver table
display(spark.table("first_data_engineering_project.silver.silver_orders").limit(10))